# 0. Import Libraries

In [ ]:
import os, sys
from pathlib import Path

def add_project_path(module_folder="model_implementation"):
    candidates = [
        os.path.abspath("."),
        os.path.abspath("../src"),
        os.path.abspath("src"),
    ]

    for path in candidates:
        if os.path.exists(os.path.join(path, module_folder)):
            if path not in sys.path:
                sys.path.append(path)
            return path

    raise ImportError(f"Could not find '{module_folder}' in current or parent directory")

SRC_PATH = add_project_path("model_implementation")
add_project_path("cnn")
add_project_path("rnn")
ROOT = Path(SRC_PATH).parent.resolve()
print("ROOT:", ROOT)

In [ ]:
# Jalankan cell ini hanya bila environment belum memiliki dependency.
# import subprocess
# subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow", "scikit-learn", "pandas", "matplotlib", "nltk", "pillow"])

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import tensorflow as tf
except Exception as exc:
    tf = None
    print("TensorFlow belum tersedia:", exc)

from cnn.utility import feature_extractor, image_paths
from common.io import save_json, load_npy
from rnn.preprocess import prep_data, load_vocab
from rnn.sequences import align_features_to_captions, teacher_pairs
from rnn.keras_models import build_preinject, compile_model
from rnn.train import grid_cfg
from rnn.weights import export_weights
from rnn.evaluate import hist_sum

# 1. Global Variables

In [ ]:
SEED = 42
IMAGE_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 10
MAX_CAPTION_LENGTH = 34

np.random.seed(SEED)
plt.style.use("seaborn-v0_8")

DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
FEATURE_DIR = DATA_DIR / "features"
VOCAB_DIR = DATA_DIR / "vocab"
MODEL_DIR = ROOT / "models"
CNN_MODEL_DIR = MODEL_DIR / "cnn"
RNN_MODEL_DIR = MODEL_DIR / "rnn"
REPORT_DIR = ROOT / "reports"
TABLE_DIR = REPORT_DIR / "tables"
FIG_DIR = REPORT_DIR / "figures"

for path in [FEATURE_DIR, VOCAB_DIR, CNN_MODEL_DIR, RNN_MODEL_DIR, TABLE_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CATEGORIES = {
    "buildings": 0,
    "forest": 1,
    "glacier": 2,
    "mountain": 3,
    "sea": 4,
    "street": 5,
}
INV_CATEGORIES = {v: k for k, v in CATEGORIES.items()}

if tf is not None:
    gpu_devices = tf.config.list_physical_devices("GPU")
    if gpu_devices:
        print("GPU Detected:", gpu_devices)
        for gpu in gpu_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
    else:
        print("No GPU found, defaulting to CPU.")

# 2. Caption Preprocessing / Loading

In [ ]:
def read_flickr8k_caption_pairs(path):
    pairs = []
    with Path(path).open('r', encoding='utf-8') as file:
        for line_number, line in enumerate(file):
            line = line.strip()
            if not line:
                continue
            if line_number == 0 and line.lower().startswith('image,'):
                continue
            if ',' in line:
                image_name, caption = line.split(',', 1)
            else:
                image_name, caption = line.split(None, 1)
            image_id = Path(image_name.split('#')[0]).stem
            pairs.append((image_id, caption))
    return pairs

CAPTION_FILE = RAW_DIR / "flickr8k" / "captions.txt"
CAPTION_PATH = VOCAB_DIR / "caption_sequences.npy"
VOCAB_PATH = VOCAB_DIR / "vocab.json"
CAPTION_ID_PATH = VOCAB_DIR / "caption_image_ids.txt"

caption_sequences = None
caption_image_ids = []
word_to_index = index_to_word = None

if CAPTION_FILE.exists():
    caption_pairs = read_flickr8k_caption_pairs(CAPTION_FILE)
    caption_image_ids = [image_id for image_id, _ in caption_pairs]
    captions = [caption for _, caption in caption_pairs]
    caption_sequences, word_to_index, index_to_word = prep_data(captions, max_length=MAX_CAPTION_LENGTH, out_dir=VOCAB_DIR)
    CAPTION_ID_PATH.write_text('\n'.join(caption_image_ids), encoding='utf-8')
    print("caption pairs:", len(caption_pairs))
    print("unique images:", len(set(caption_image_ids)))
    print("sequences:", caption_sequences.shape)
    print("vocab:", len(word_to_index))
elif CAPTION_PATH.exists() and VOCAB_PATH.exists() and CAPTION_ID_PATH.exists():
    caption_sequences = load_npy(CAPTION_PATH).astype('int32')
    caption_image_ids = [line.strip() for line in CAPTION_ID_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
    word_to_index, index_to_word = load_vocab(VOCAB_DIR)
    print("Loaded caption artifacts:", caption_sequences.shape, len(caption_image_ids), len(word_to_index))
else:
    print("Caption artifacts belum tersedia.")

# 3. Feature Extraction / Loading

In [ ]:
IMAGE_DIR = RAW_DIR / "flickr8k" / "Images"
FEATURE_PATH = FEATURE_DIR / "flickr8k_features.npy"
FEATURE_ID_PATH = FEATURE_DIR / "flickr8k_image_ids.txt"
ENCODER_PATH = CNN_MODEL_DIR / "flickr8k_inceptionv3_encoder.keras"

features = None
feature_image_ids = []

if tf is not None and IMAGE_DIR.exists():
    img_paths = image_paths(IMAGE_DIR)
    feature_image_ids = [path.stem for path in img_paths]
    if ENCODER_PATH.exists():
        encoder = tf.keras.models.load_model(ENCODER_PATH)
    else:
        encoder = tf.keras.applications.InceptionV3(include_top=False, weights='imagenet', input_shape=(299, 299, 3), pooling='avg')
        encoder.trainable = False
        encoder.save(ENCODER_PATH)
    features = feature_extractor(img_paths, encoder, FEATURE_PATH, target_size=encoder.input_shape[1:3], batch_size=BATCH_SIZE, image_id_path=FEATURE_ID_PATH, preprocess_fn=tf.keras.applications.inception_v3.preprocess_input)
    print("images:", len(feature_image_ids))
    print("features:", features.shape)
elif FEATURE_PATH.exists() and FEATURE_ID_PATH.exists():
    features = load_npy(FEATURE_PATH).astype('float32')
    feature_image_ids = [line.strip() for line in FEATURE_ID_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
    print("Loaded existing features:", features.shape, len(feature_image_ids))
else:
    print("Feature extraction dilewati.")

# 4. RNN/LSTM Decoder Training

In [ ]:
def split_arrays(features, captions, val_ratio=0.2):
    rng = np.random.default_rng(SEED)
    order = np.arange(features.shape[0])
    rng.shuffle(order)
    val_count = int(round(len(order) * val_ratio))
    val_idx, train_idx = order[:val_count], order[val_count:]
    return (features[train_idx], captions[train_idx]), (features[val_idx], captions[val_idx])

def train_decoder_model(model, train_features, train_captions, val_data=None, batch_size=32, epochs=10):
    train_inputs, train_targets = teacher_pairs(train_captions)
    validation_data = None
    if val_data is not None:
        val_features, val_captions = val_data
        val_inputs, val_targets = teacher_pairs(val_captions)
        validation_data = ([val_features, val_inputs], val_targets)
    return model.fit([train_features, train_inputs], train_targets, validation_data=validation_data, batch_size=batch_size, epochs=epochs, verbose=1)

aligned_features = None
aligned_captions = None
aligned_caption_ids = []
rnn_records = []

if features is not None and caption_sequences is not None and caption_image_ids and feature_image_ids:
    aligned_features, aligned_captions, aligned_caption_ids, missing_caption_ids = align_features_to_captions(
        features, feature_image_ids, caption_sequences, caption_image_ids
    )
    if missing_caption_ids:
        print('caption rows without image feature:', len(missing_caption_ids))
    print('aligned features/captions:', aligned_features.shape, aligned_captions.shape)

if tf is not None and aligned_features is not None and aligned_captions is not None and word_to_index is not None:
    base_config = {
        'vocab_size': len(word_to_index),
        'feature_dim': int(aligned_features.shape[1]),
        'max_length': int(aligned_captions.shape[1]),
        'caption_length': int(aligned_captions.shape[1] - 1),
        'embed_dim': 256,
        'learning_rate': 1e-3,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
    }
    train_data, val_data = split_arrays(aligned_features, aligned_captions)
    train_features, train_captions = train_data

    for cfg in grid_cfg(base_config):
        print("\n--- Training decoder:", cfg['model_name'], "---")
        model = build_preinject(
            vocab_size=cfg['vocab_size'], feature_dim=cfg['feature_dim'], max_length=cfg['caption_length'],
            embed_dim=cfg['embed_dim'], hidden_size=cfg['hidden_size'], recur_layers=cfg['recur_layers'], recur_type=cfg['recur_type']
        )
        model = compile_model(model, learn_rate=cfg['learning_rate'])
        history = train_decoder_model(model, train_features, train_captions, val_data=val_data, batch_size=cfg['batch_size'], epochs=cfg['epochs'])
        model_path = RNN_MODEL_DIR / cfg['model_name']
        weight_path = RNN_MODEL_DIR / f"{Path(cfg['model_name']).stem}.npz"
        history_path = TABLE_DIR / f"{Path(cfg['model_name']).stem}_history.json"
        model.save(model_path)
        export_weights(model, weight_path)
        save_json(hist_sum(history), history_path)
        rnn_records.append({'config': cfg, 'model_path': str(model_path), 'weight_path': str(weight_path), 'history_path': str(history_path)})

    save_json(rnn_records, TABLE_DIR / 'train_records.json')
    (FEATURE_DIR / 'aligned_caption_image_ids.txt').write_text('\n'.join(aligned_caption_ids), encoding='utf-8')
else:
    print("RNN/LSTM training dilewati.")

pd.DataFrame(rnn_records)